In [ ]:
# 🤖 Logistic Regression — Risk Classification & Safe Price Estimation

import os
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay



# 2️⃣ Ensure risky_label exists

if "risky_label" not in feature_df.columns:
    spike_threshold = feature_df["spike_count"].quantile(0.9)
    vol_threshold   = feature_df["volatility"].quantile(0.9)
    feature_df["risky_label"] = np.where(
        (feature_df["spike_count"] >= spike_threshold) |
        (feature_df["volatility"]   >= vol_threshold),
        1, 0
    )


# 3️⃣ Prepare features and labels

X = feature_df[[
    "mean_price_usd",
    "mean_pct_change",
    "volatility",
    "mean_deviation",
    "spike_count"
]].copy()
y = feature_df["risky_label"].values

# Clean data
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
X = np.clip(X, -1e6, 1e6)
X = np.array(X, dtype=float)

# Split
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale
scaler = MinMaxScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled  = scaler.transform(x_test)

# 4️⃣ Train Logistic Regression

print("🚀 Training Logistic Regression model ...")
log_reg = LogisticRegression(max_iter=1000, C=1.0)
log_reg.fit(x_train_scaled, y_train)
print("✅ Training complete!")


# 5️⃣ Evaluate Model

y_pred = log_reg.predict(x_test_scaled)
print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred))
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Purples")
plt.title("Logistic Regression Confusion Matrix")
plt.show()


# 7️⃣ Predict Risk and Safe Prices

X_scaled = scaler.transform(X)
risk_prob = log_reg.predict_proba(X_scaled)[:, 1]

feature_df["risk_score_LogReg"] = risk_prob
alpha = 0.25  # same discount rate
feature_df["safe_price_usd_LogReg"] = feature_df["mean_price_usd"] * (1 - alpha * risk_prob)


# 8️⃣ Save Report

os.makedirs("../data/report", exist_ok=True)
logreg_report = feature_df[[
    "mean_price_usd", "volatility", "spike_count",
    "risk_score_LogReg", "safe_price_usd_LogReg"
]].sort_values(by="risk_score_LogReg", ascending=False)

logreg_report.to_csv("../data/report/logistic_safe_prices.csv", index=True)
print("💾 Saved → ../data/report/logistic_safe_prices.csv")


# 9️⃣ Visualize Safe vs Market Price

plt.figure(figsize=(10,6))
plt.scatter(logreg_report["mean_price_usd"], logreg_report["safe_price_usd_LogReg"],
            c=logreg_report["risk_score_LogReg"], cmap="Purples", alpha=0.7)
plt.colorbar(label="Predicted Risk Probability (0–1)")
plt.xlabel("Market Price (USD)")
plt.ylabel("Safe Price (USD)")
plt.title("Logistic Regression — Safe vs Market Prices")
plt.grid(True)
plt.tight_layout()
plt.show()